## Phase 0.2 – Effect of the train/test split on the late delivery model

The same Random Forest classifier is trained and evaluated with three different splits:

1. **Random rows (original approach):** 80/20 stratified random split of order items.
2. **Grouped by Order Id:** all items of an order are placed either in the training or in the test set.
3. **Time-based:** the model is trained on the oldest 80% of orders and tested on the most recent 20%.

Items belonging to the same order almost always share the same label. With a random row split, items of one order end up in both sets, so the model can effectively "recognise" the order instead of learning a generalisable pattern (data leakage). The time-based split reflects real use: the model learns from the past and predicts the future. Therefore, the time-based result is reported as the realistic performance estimate.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

In [2]:
from src.data import load_data

data = load_data()
print(data.shape)

(180519, 51)


In [3]:
feature_cols = ["Customer Zipcode", "order_year", "order_month", "order_week_day", "order_hour"]
cat_cols = ["Shipping Mode", "Market", "Order Region", "Category Name",
            "Department Name", "Customer Segment", "Type"]

X = pd.get_dummies(data[feature_cols + cat_cols], columns=cat_cols, drop_first=True)
y = data["Late_delivery_risk"]
groups = data["Order Id"]   # melyik sor melyik rendeléshez tartozik
print("Features:", X.shape[1])

Features: 98


In [4]:
def evaluate(train_idx, test_idx, name):
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"split": name,
            "accuracy": round(accuracy_score(y.iloc[test_idx], pred), 3),
            "recall_late": round(recall_score(y.iloc[test_idx], pred), 3),
            "roc_auc": round(roc_auc_score(y.iloc[test_idx], proba), 3)}

all_idx = np.arange(len(data))
results = []

tr, te = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y)
results.append(evaluate(tr, te, "1) random rows (old)"))

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
results.append(evaluate(tr, te, "2) grouped by Order Id"))

cut = int(len(data) * 0.8)
results.append(evaluate(all_idx[:cut], all_idx[cut:], "3) time-based (last 20%)"))

print(pd.DataFrame(results).to_string(index=False))
print("Time split boundary:", data.loc[cut, "order_date"])

                   split  accuracy  recall_late  roc_auc
    1) random rows (old)     0.806        0.788    0.896
  2) grouped by Order Id     0.682        0.667    0.750
3) time-based (last 20%)     0.695        0.657    0.760
Time split boundary: 2017-04-22 15:17:00
